# SpaceX Falcon 9 — Launch Records Dashboard (Plotly Dash)

The full interactive dashboard app lives in
[`dashboard/spacex_dash_app.py`](./dashboard/spacex_dash_app.py) — run it locally with
`python spacex_dash_app.py` and open `http://127.0.0.1:8050`. It provides:

* a dropdown to pick a launch site (or **All Sites**)
* a pie chart of launch success counts for the selection
* a range slider on payload mass (kg)
* a scatter plot of payload vs. launch outcome, colored by booster version, for the selected site + payload range

This notebook renders the same three figures with Plotly so the outputs are visible without running a local server.

In [1]:
import pandas as pd
import plotly.express as px

BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"
spacex_df = pd.read_csv(f"{BASE}/datasets/spacex_launch_dash.csv")
spacex_df.head()

,Unnamed: 0,Flight Number,Launch Site,class,Payload Mass (kg),Booster Version,Booster Version Category
0,0,1,CCAFS LC-40,0,0.0,F9 v1.0 B0003,v1.0
1,1,2,CCAFS LC-40,0,0.0,F9 v1.0 B0004,v1.0
2,2,3,CCAFS LC-40,0,525.0,F9 v1.0 B0005,v1.0
3,3,4,CCAFS LC-40,0,500.0,F9 v1.0 B0006,v1.0
4,4,5,CCAFS LC-40,0,677.0,F9 v1.0 B0007,v1.0


## Pie chart — total successful launches by site (all sites)

In [2]:
success_by_site = spacex_df[spacex_df['class'] == 1]['Launch Site'].value_counts().reset_index()
success_by_site.columns = ['Launch Site', 'Successful Launches']
fig1 = px.pie(success_by_site, values='Successful Launches', names='Launch Site',
              title='Total Successful Launches by Site')
fig1.show()

## Pie chart — success vs. failure ratio for the best-performing site

In [3]:
best_site = spacex_df.groupby('Launch Site')['class'].mean().idxmax()
site_df = spacex_df[spacex_df['Launch Site'] == best_site]
counts = site_df['class'].value_counts().rename({1: 'Success', 0: 'Failure'}).reset_index()
counts.columns = ['Outcome', 'Count']
fig2 = px.pie(counts, values='Count', names='Outcome', title=f'Launch Outcomes for {best_site} (highest success ratio)',
              color='Outcome', color_discrete_map={'Success': '#24A148', 'Failure': '#DA1E28'})
fig2.show()
print(f"Best-performing site: {best_site}  (success rate {site_df['class'].mean():.0%})")

Best-performing site: KSC LC-39A  (success rate 77%)


## Payload vs. launch outcome, colored by booster version category

Shown for the full payload range and for boosters between 2,000–6,000 kg (an example range-slider selection).

In [4]:
fig3 = px.scatter(spacex_df, x='Payload Mass (kg)', y='class', color='Booster Version Category',
                   title='Payload vs. Outcome — All Sites, Full Payload Range',
                   labels={'class': 'Launch Outcome (0 = Failure, 1 = Success)'})
fig3.show()

sel = spacex_df[(spacex_df['Payload Mass (kg)'] >= 2000) & (spacex_df['Payload Mass (kg)'] <= 6000)]
fig4 = px.scatter(sel, x='Payload Mass (kg)', y='class', color='Booster Version Category',
                   title='Payload vs. Outcome — All Sites, 2,000-6,000 kg Range Slider Selection',
                   labels={'class': 'Launch Outcome (0 = Failure, 1 = Success)'})
fig4.show()